# J04 — Écoulement de l'eau en sol saturé

**Physique et hydrodynamique des sols** — S. J. Gumiere

*Atelier du Jour 4 : perméamètres, sol stratifié, écartement de drains (Hooghoudt), Laplace 2D par différences finies*

> Version **étudiant** : complétez les cellules marquées `# À COMPLÉTER`.

## Mise en place

Unités : cm et jours pour les sols (convention du cours et de HYDRUS-1D), m et jours pour le drainage et Laplace 2D.
$z$ positif vers le haut, $q > 0$ vers le haut, $H = h + z$.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import optimize, sparse
from scipy.sparse.linalg import spsolve

plt.rcParams.update({"figure.figsize": (7, 4), "axes.grid": True, "grid.alpha": 0.3})
RHO_W, G = 998.2, 9.81                     # kg/m³, m/s²
T_TAB  = np.array([5, 10, 15, 18, 20, 25, 30])
MU_TAB = np.array([1.519, 1.307, 1.139, 1.053, 1.002, 0.890, 0.798]) * 1e-3   # Pa s
viscosite = lambda T: np.interp(T, T_TAB, MU_TAB)

## Exercice 1 — Perméamètres à charge constante et à charge variable  *(≈ 15 min)*

1. `data/J04_permeametre_charge_constante.csv` : quatre échantillons (S1–S4) mesurés trois fois chacun à 18 °C
   ($L$, $A$, $\Delta H$, volume $V$ recueilli en $t$). Calculer $K_s = V L/(A\,\Delta H\,t)$ pour chaque essai (cm/s → cm/j et m/s),
   puis la moyenne et l'écart type par échantillon ; ramener à 20 °C : $K_s(20) = K_s(T)\,\mu(T)/\mu(20)$.
2. `data/J04_permeametre_charge_variable.csv` : lectures $h(t)$ pour deux échantillons (colonnes `a_cm2`, `A_cm2`, `L_cm`).
   Ajuster $\ln(h_0/h) = (A K_s/aL)\,t$ par régression linéaire (`np.polyfit(..., cov=True)`) ; en déduire $K_s$ et son erreur type ;
   tracer les points et la droite.
3. Comparer les $K_s$ mesurées à l'estimation de Kozeny–Carman, $K_s = (\rho_w g/\mu)\, n^3 d^2/[180(1-n)^2]$ avec $d = d_{10}$,
   pour S1–S4 (colonnes `d10_mm`, `n`). Commenter les écarts.

In [ ]:
cc = pd.read_csv("data/J04_permeametre_charge_constante.csv")
cv = pd.read_csv("data/J04_permeametre_charge_variable.csv")

# 1. charge constante
cc["Ks_cms"] = # À COMPLÉTER
cc["Ks_20_cmj"] = # À COMPLÉTER : correction de viscosité puis cm/s -> cm/j
res_cc = cc.groupby("echantillon")["Ks_20_cmj"].agg(["mean", "std"])

# 2. charge variable : régression de ln(h0/h) sur t
def ks_charge_variable(g):
    t, h = g["t_s"].to_numpy(), g["h_cm"].to_numpy()
    a, A, L = g["a_cm2"].iloc[0], g["A_cm2"].iloc[0], g["L_cm"].iloc[0]
    # À COMPLÉTER : pente p (1/s) et son erreur type ; Ks = p * a * L / A
    pass

# 3. Kozeny-Carman
def kozeny_carman(d_m, n, T=20):
    # À COMPLÉTER
    pass

## Exercice 2 — Sol à trois couches : flux et profils de charge  *(≈ 15 min)*

Sable ($K_1 = 50$ cm/j, 30 cm) sur argile ($K_2 = 0{,}5$ cm/j, 10 cm) sur sable ($K_3 = 20$ cm/j, 60 cm) ; lame d'eau de 5 cm en
surface ($H_{haut} = 5$ cm) ; nappe à la base ($z = -100$ cm, $h = 0$, $H_{bas} = -100$ cm).

1. Écrire `profil_serie(K, L, H_haut, H_bas)` qui retourne $K_\perp = \sum L_i / \sum (L_i/K_i)$, le flux $q = -K_\perp \Delta H/\sum L_i$
   et les tableaux $z$, $H$, $h$ aux interfaces (en supposant tout saturé).
2. Tracer $H(z)$ et $h(z)$ ; où la pression devient-elle négative ? Quelle fraction de la perte de charge a lieu dans l'argile ?
3. Nappe perchée : recalculer le flux en imposant $h = 0$ à la base de l'argile ($H = -40$ cm) sur les deux couches supérieures ;
   comparer au cas « tout saturé ».
4. Remplacer l'argile par un loam ($K_2 = 25$ cm/j) : le profil reste-t-il saturé ? Flux ?

In [ ]:
def profil_serie(K, L, H_haut, H_bas):
    """Écoulement vertical permanent à travers des couches en série (tout saturé).
    Retourne K_eq (cm/j), q (cm/j, > 0 vers le haut), z, H, h aux interfaces (cm)."""
    K, L = np.asarray(K, float), np.asarray(L, float)
    K_eq = # À COMPLÉTER
    q = # À COMPLÉTER
    z = np.concatenate([[0.0], -np.cumsum(L)])
    H = # À COMPLÉTER : H_haut puis pertes de charge successives |q| L_i / K_i
    h = H - z
    return K_eq, q, z, H, h

K_eq, q, z, H, h = profil_serie([50, 0.5, 20], [30, 10, 60], 5.0, -100.0)

## Exercice 3 — Écartement de drains : Hooghoudt et profondeur équivalente  *(≈ 15 min)*

Équation de Hooghoudt : $L^2 = (8 K_b d_e h + 4 K_a h^2)/q$ (m, j), avec la profondeur équivalente de van der Molen & Wesseling (1991) :

$d_e = \dfrac{\pi L/8}{\ln[L/(\pi r_0)] + F(x)}$, $x = 2\pi D/L$, $F(x) = \sum_{j=1,3,5,\dots} \dfrac{4 e^{-2jx}}{j(1-e^{-2jx})}$ pour $x > 0{,}5$
et $F(x) \approx \pi^2/(4x) + \ln[x/(2\pi)]$ pour $x < 0{,}5$.

1. Programmer `de_vdmw(L, D, r0)` ; vérifier : $L = 20$ m, $D = 1$ m, $r_0 = 0{,}1$ m → $d_e \approx 0{,}87$ m ; $D = 1000$ m → $\approx 1{,}89$ m ;
   $D = 0{,}5$ m → $\approx 0{,}49$ m ($\approx D$).
2. Programmer `ecartement_drains(q, K, h, D, r0, Ka=None)` (itération de point fixe : $L \to d_e \to L$) ; retrouver
   $L = 28{,}3$ m pour $q = 7$ mm/j, $K = 0{,}8$ m/j, $h = 0{,}5$ m, $D = 2$ m, $r_0 = 0{,}1$ m.
3. Table de $L$ pour $K \in \{0{,}2 ; 0{,}5 ; 1 ; 2\}$ m/j et $D \in \{0{,}5 ; 1 ; 2 ; 5 ; 1000\}$ m ; graphique $L(K)$ pour chaque $D$.
4. Sensibilité (cas de référence) : variation relative de $L$ pour $K \times 0{,}5$ et $\times 2$, et pour $h = 0{,}4$ et $0{,}6$ m.

In [ ]:
def de_vdmw(L, D, r0):
    """Profondeur équivalente de Hooghoudt (van der Molen & Wesseling 1991), unités cohérentes (m)."""
    x = 2 * np.pi * D / L
    # À COMPLÉTER
    pass

def ecartement_drains(q, K, h, D, r0=0.1, Ka=None, tol=1e-6):
    """Écartement L (m) par itération de point fixe sur d_e."""
    Ka = K if Ka is None else Ka
    L = 30.0
    # À COMPLÉTER
    return L

print(de_vdmw(20, 1, 0.1), de_vdmw(20, 1000, 0.1), de_vdmw(20, 0.5, 0.1))

## Exercice 4 — Laplace 2D par différences finies : écoulement sous une palplanche  *(≈ 15 min)*

Domaine $30 \times 15$ m (grille $61 \times 31$, $\Delta = 0{,}5$ m), $K = 1$ m/j, palplanche en $x = 15$ m fichée à 6 m (elle coupe la
connexion entre les colonnes $i_p - 1$ et $i_p$ jusqu'à $z = -6$ m). Surface : $H = 4$ m à l'amont ($x < 15$), $H = 0$ à l'aval ;
parois latérales et fond imperméables (Neumann : nœud miroir).

1. Construire les tableaux d'indices des quatre voisins de chaque nœud (avec les miroirs) et implémenter un solveur
   Gauss–Seidel **vectorisé** par balayage en damier rouge–noir, avec sur-relaxation ($\omega = 1{,}8$) et critère
   $\max|\Delta H| < 10^{-6}$ m. Combien d'itérations ?
2. Comparer à la solution directe (matrice creuse + `spsolve`) : écart maximal.
3. Tracer les équipotentielles et le champ de flux ($q_x = -K\,\partial H/\partial x$, $q_z = -K\,\partial H/\partial z$).
4. Calculer le débit de fuite $Q$ (m³/j par m) par la surface aval, le débit entrant à l'amont (bilan), le facteur de forme
   $Q/(K\Delta H) = N_f/N_d$ et le gradient hydraulique de sortie à l'aval, contre la palplanche (à comparer au gradient critique $\approx 1$).

In [ ]:
nx, nz, dx = 61, 31, 0.5
Kh = 1.0; H_am, H_av = 4.0, 0.0
ip, zp = 30, 12                       # colonne de la palplanche ; profondeur (indices de ligne 0..zp)

def voisins(nx, nz, ip, zp):
    """Indices (ligne, colonne) des voisins gauche/droite/haut/bas avec miroirs (Neumann) et palplanche."""
    J, I = np.meshgrid(np.arange(nz), np.arange(nx), indexing="ij")
    Il = np.where(I > 0, I - 1, I + 1); Ir = np.where(I < nx - 1, I + 1, I - 1)
    Ju = np.where(J > 0, J - 1, J + 1); Jd = np.where(J < nz - 1, J + 1, J - 1)
    # À COMPLÉTER : palplanche (lignes 0..zp) : le voisin droit de ip-1 est ip-2, le voisin gauche de ip est ip+1
    return (J, Il), (J, Ir), (Ju, I), (Jd, I)

def sor(H, vois, libre, omega=1.8, tol=1e-6, itmax=50000):
    # À COMPLÉTER : damier rouge-noir, mise à jour H = H + omega*(Hgs - H) sur les noeuds libres
    pass

## Exercice 5 — Bonus — milieu anisotrope  *(≈ facultatif)*

Refaire l'exercice 4 avec $K_h = 4$ m/j et $K_v = 1$ m/j : le schéma devient
$H_{i,j} = [K_h(H_{i+1,j} + H_{i-1,j}) + K_v(H_{i,j+1} + H_{i,j-1})]/[2(K_h + K_v)]$ ($\Delta x = \Delta z$).
Comparer le débit de fuite (avec $q_x = -K_h\,\partial H/\partial x$, $q_z = -K_v\,\partial H/\partial z$) et l'allure des
équipotentielles au cas isotrope. Vérifier avec la transformation $x' = x\sqrt{K_v/K_h}$ : $Q_{aniso} \approx \sqrt{K_h K_v}\,\Delta H\,(N_f/N_d)'$.

In [ ]:
Kh2, Kv2 = 4.0, 1.0
# À COMPLÉTER : adapter sor() avec les poids Kh, Kv

## Pour aller plus loin

* Exercice 2 : traiter le cas non saturé sous l'argile avec $K(h)$ de van Genuchten–Mualem (Jour 5) et comparer au flux de la nappe perchée.
* Exercice 3 : ajouter la formule non permanente de Glover–Dumm pour la vidange après une pluie de 40 mm.
* Exercice 4 : raffiner la grille ($\Delta = 0{,}25$ m) et étudier la convergence du gradient de sortie ; ajouter un radier amont.